# Positional Encoding

Adding this to the attention notebook because attention on its own has no sense of order — it treats the input as a set. Two sentences with the same words in different order would produce identical representations. That's obviously wrong.

The fix from the original paper: add a positional signal directly to the input embeddings before passing them into the transformer.

## The Math

For each position `pos` and each dimension `i` in the embedding:

```
PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))
```

Even dimensions get sine, odd dimensions get cosine.

### Why sinusoids?

A few reasons the paper gives:

1. **Generalises to unseen sequence lengths** — sinusoids are defined for any position, so the model can handle sequences longer than anything seen during training.

2. **Relative positions are learnable** — for any fixed offset k, PE(pos+k) can be expressed as a linear function of PE(pos). This means the model can learn to attend by relative position.

3. **Bounded values** — always between -1 and 1, so it doesn't distort the embeddings.

The wavelengths form a geometric progression from 2π to 10000·2π. Low dimensions capture fine-grained position (high frequency), high dimensions capture coarse position (low frequency) — similar to how binary numbers work.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## Implementation

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # build the PE matrix once — shape (max_seq_len, d_model)
        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len).unsqueeze(1).float()  # (max_seq_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dims

        # add batch dim and register as buffer (not a parameter — not trained)
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# sanity check
pe = PositionalEncoding(d_model=512, dropout=0.0)
x  = torch.zeros(1, 100, 512)
out = pe(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")  # same — PE is additive
print(f"PE is not all zeros: {not out.allclose(x)}")

## Visualise It

This is worth looking at — you can actually see the sinusoidal patterns across positions and dimensions.

In [ ]:
pe_vis = PositionalEncoding(d_model=128, dropout=0.0)
x_vis  = torch.zeros(1, 100, 128)
pe_matrix = pe_vis(x_vis).squeeze(0).detach().numpy()  # (100, 128)

plt.figure(figsize=(12, 5))
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.xlabel('Position in sequence')
plt.ylabel('Embedding dimension')
plt.title('Sinusoidal Positional Encoding — each row is a dimension, each col is a position')
plt.colorbar()
plt.tight_layout()
plt.show()

## Plug It Into the Encoder

Now adding PE to the full encoder from the previous notebook.

In [ ]:
# pasting in the components from attention-from-scratch.ipynb
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    return torch.matmul(F.softmax(scores, dim=-1), V)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_k       = d_model // num_heads
        self.num_heads = num_heads
        self.d_model   = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        b, s, _ = x.shape
        return x.view(b, s, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        b, _, s, _ = x.shape
        return x.transpose(1, 2).contiguous().view(b, s, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        out = scaled_dot_product_attention(Q, K, V, mask)
        return self.W_o(self.combine_heads(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ff      = FeedForward(d_model, d_ff)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    """
    Full encoder: embedding + positional encoding + N stacked encoder blocks.
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe        = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers    = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # x: (batch, seq_len) — token ids
        x = self.pe(self.embedding(x))
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


# full encoder sanity check
encoder = TransformerEncoder(
    vocab_size=1000, d_model=512, num_heads=8,
    d_ff=2048, num_layers=6
)

tokens = torch.randint(0, 1000, (4, 20))  # (batch=4, seq_len=20)
out    = encoder(tokens)
print(f"Encoder output: {out.shape}")  # (4, 20, 512)
print(f"Parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## Notes

A few things I found interesting working through this:

- PE is registered as a **buffer**, not a parameter — meaning it's not trained, just stored with the model. That's the right call since it's a fixed function of position.
- The `div_term` uses `exp(log(...))` instead of a direct power — numerically more stable.
- Modern models (LLaMA, GPT-NeoX) use **RoPE** (Rotary Positional Embedding) instead of additive sinusoidal PE. RoPE encodes position by rotating the query and key vectors — it handles relative positions more naturally. Worth reading next.

At this point the full encoder is done from scratch: embedding → positional encoding → N × (multi-head attention + FFN + residuals + layer norm). That's the left half of the original Transformer paper.